In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers accelerate sentencepiece pandas wandb huggingface_hub bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.5 MB/s eta 0:00:00


In [ ]:
import json
import re
import pandas as pd
import torch
import wandb

from pathlib import Path
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

In [ ]:
from huggingface_hub import login
login("<HF_TOKEN>")

wandb.login(key="<WANDB_API_KEY>")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nishkala-aistuff (nishkala-aistuff-georgia-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2"

BENCHMARK_PATH = (
    f"{PROJECT_ROOT}/eval/llm_only/benchmarks/llm_only_v1.json"
)

RUNS_DIR = (
    f"{PROJECT_ROOT}/eval/llm_only/runs/model_comparison"
)

OUTPUT_CSV = (
    f"{PROJECT_ROOT}/eval/llm_only/scores/judge_scores_v1.csv"
)

JUDGE_MODEL = "Qwen/Qwen2.5-14B-Instruct"

WANDB_PROJECT = "ARIA-Lite-Eval"
WANDB_GROUP = "llm_judge_v1"

In [ ]:
with open(BENCHMARK_PATH, "r") as f:
    benchmark = json.load(f)

benchmark_lookup = {
    item["query_id"]: item
    for item in benchmark
}

print("Loaded benchmark:", len(benchmark_lookup))

Loaded benchmark: 4


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    JUDGE_MODEL
)

model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()

print("Judge loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Judge loaded.


In [ ]:
def build_judge_prompt(
    query,
    sections,
    reference_answer,
    candidate_answer
):

    return f"""
You are an expert biomedical evaluator.

Evaluate the candidate answer.

SCORING RUBRIC

Correctness (0-5):
How factually accurate is the answer relative to the provided evidence?

Completeness (0-5):
How completely does the answer cover the major findings?

Grounding (0-5):
How well is the answer supported by the provided sections?

Biomedical Soundness (0-5):
Are biological mechanisms, terminology, and interpretations scientifically sound?

Return ONLY this format:

CORRECTNESS: X
COMPLETENESS: X
GROUNDING: X
BIOMEDICAL_SOUNDNESS: X
TOTAL: X

REASONING:
<brief explanation>

--------------------------------

QUERY:
{query}

--------------------------------

SECTIONS:
{sections}

--------------------------------

REFERENCE ANSWER:
{reference_answer}

--------------------------------

CANDIDATE ANSWER:
{candidate_answer}
"""

In [ ]:
def judge_answer(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=12000
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        temperature=0.0
    )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

In [ ]:
def extract_score(text, key):

    pattern = rf"{key}\s*:\s*(\d+)"

    match = re.search(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    if match:
        return int(match.group(1))

    return None

In [ ]:
wandb.init(
    project=WANDB_PROJECT,
    group=WANDB_GROUP,
    name="Qwen2.5-14B-Instruct_judge"
)

In [ ]:
results = []

for model_dir in Path(RUNS_DIR).iterdir():

    if not model_dir.is_dir():
        continue

    model_name = model_dir.name

    print(f"\nEvaluating: {model_name}")

    for json_file in model_dir.glob("*.json"):

        with open(json_file, "r") as f:
            run_data = json.load(f)

        query_id = run_data["query_id"]

        benchmark_item = benchmark_lookup[query_id]

        sections = "\n\n".join(
            [
                section["text"]
                for section in benchmark_item["sections"]
            ]
        )

        prompt = build_judge_prompt(
            query=benchmark_item["query"],
            sections=sections,
            reference_answer=benchmark_item["reference_answer"],
            candidate_answer=run_data["response"]
        )

        judgment = judge_answer(prompt)

        correctness = extract_score(
            judgment,
            "CORRECTNESS"
        )

        completeness = extract_score(
            judgment,
            "COMPLETENESS"
        )

        grounding = extract_score(
            judgment,
            "GROUNDING"
        )

        biomedical = extract_score(
            judgment,
            "BIOMEDICAL_SOUNDNESS"
        )

        total = extract_score(
            judgment,
            "TOTAL"
        )

        row = {
            "query_id": query_id,
            "model": model_name,
            "correctness": correctness,
            "completeness": completeness,
            "grounding": grounding,
            "biomedical_soundness": biomedical,
            "total": total,
            "judge_output": judgment
        }

        results.append(row)

        wandb.log(row)


Evaluating: qwen1.5b


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Evaluating: qwen3b

Evaluating: mistral7b


In [ ]:
scores_df = pd.DataFrame(results)

scores_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print(
    "Saved:",
    OUTPUT_CSV
)

scores_df.head()

Saved: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/llm_only/scores/judge_scores_v1.csv


,query_id,model,correctness,completeness,grounding,biomedical_soundness,total,judge_output
0,Q1,qwen1.5b,4,3,4,4,4,CORRECTNESS: 4\nCOMPLETENESS: 3\nGROUNDING: 4\...
1,Q2,qwen1.5b,4,4,4,5,17,CORRECTNESS: 4\nCOMPLETENESS: 4\nGROUNDING: 4\...
2,Q3,qwen1.5b,4,4,4,5,4,CORRECTNESS: 4\nCOMPLETENESS: 4\nGROUNDING: 4\...
3,Q4,qwen1.5b,4,4,4,5,4,CORRECTNESS: 4\nCOMPLETENESS: 4\nGROUNDING: 4\...
4,Q1,qwen3b,4,4,4,5,4,CORRECTNESS: 4\nCOMPLETENESS: 4\nGROUNDING: 4\...


In [ ]:
leaderboard = (
    scores_df
    .groupby("model")["total"]
    .agg(["mean", "std", "min", "max"])
    .sort_values("mean", ascending=False)
)

leaderboard

,mean,std,min,max
model,,,,
qwen1.5b,7.25,6.5,4,17
mistral7b,4.00,0.0,4,4
qwen3b,4.00,0.0,4,4


In [ ]:
wandb.log(
    {
        "leaderboard":
        wandb.Table(
            dataframe=leaderboard.reset_index()
        )
    }
)

wandb.finish()

biomedical_soundness,▁███████▁███
completeness,▁███████████
correctness,▁▁▁▁▁▁▁▁▁▁▁▁
grounding,▁▁▁▁▁▁▁▁▁▁▁█
total,▁█▁▁▁▁▁▁▁▁▁▁
biomedical_soundness,5
completeness,4
correctness,4
grounding,5
judge_output,CORRECTNESS: 4COMPL...
model,mistral7b
